In [ ]:
# ============================================================
# ROGII v13 — INFERENCE NOTEBOOK
# ============================================================
# This notebook:
#   • Loads imputers from artifacts (no training data rebuild)
#   • Builds ONLY test features (~35 min for hidden test set)
#   • Loads trained models fold by fold and predicts
#   • Applies saved HillClimber weights
#   • Applies saved PP params (alpha, tau, w_pf)
#   • Generates submission.csv
#
# ============================================================

from __future__ import annotations
import subprocess, sys, os

for pkg in ["numba","catboost","pywavelets"]:
    if subprocess.run([sys.executable,"-m","pip","show",pkg],capture_output=True).returncode!=0:
        subprocess.run([sys.executable,"-m","pip","install",pkg,"--quiet"])

os.environ["NUMBA_CACHE_DIR"]="/kaggle/working/.numba"
os.makedirs("/kaggle/working/.numba",exist_ok=True)

from pathlib import Path
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from numba import njit
from joblib import Parallel, delayed
from catboost import CatBoostRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
import lightgbm as lgb
import pywt
import numpy as np, pandas as pd
import gc, time, joblib, multiprocessing, warnings

warnings.filterwarnings("ignore")
SEED=42; np.random.seed(SEED)
NCPU=min(4,multiprocessing.cpu_count())

# ── Paths ──────────────────────────────────────────────────────────────────────
def _find_data():
    for p in [Path("/kaggle/input/rogii-wellbore-geology-prediction"),
              Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")]:
        if (p/"train").exists(): return p
    raise FileNotFoundError("Competition data not found")

def _find_arts():
    # Only models/ and meta/ are required — no imputers/ needed anymore
    for p in [Path("/kaggle/input/datasets/tasmim/rogii-v13-artifacts/rogii_v13_artifacts"),
              Path("/kaggle/working/rogii_v13_artifacts")]:
        if (p/"models").exists(): return p
    raise FileNotFoundError(
        "Artifacts not found. Upload rogii_v13_artifacts/ as Kaggle dataset "
        "'rogii-v13-artifacts' (must contain models/ and meta/ directories).")

DATA     = _find_data()
ARTS     = _find_arts()
TRAIN_DIR = DATA/"train"   # for building imputers from scratch
TEST_DIR  = DATA/"test"
SAMPLE    = DATA/"sample_submission.csv"
OUT       = Path("/kaggle/working/submission.csv")

print(f"Data:       {DATA}")
print(f"Artifacts:  {ARTS}")
print(f"Train wells: {len(list(TRAIN_DIR.glob('*__horizontal_well.csv')))}")
print(f"Test wells:  {len(list(TEST_DIR.glob('*__horizontal_well.csv')))}")

# ── Constants (must match training notebook exactly) ───────────────────────────
FORMATIONS=["ANCC","ASTNU","ASTNL","EGFDU","EGFDL","BUDA"]
PLANE_K=10; DENSE_SPW=60; DENSE_K=20; N_SPLITS=5

ANCH_OFFS=np.array([-80,-40,-20,-10,-5,0,5,10,20,40,80],np.float32)
BEAM_OFFS=np.array([-40,-20,-10,-5,-3,0,3,5,10,20,40],np.float32)
SC_OFFS  =np.array([-30,-15,-8,-4,-2,0,2,4,8,15,30],np.float32)
PF_OFFS  =np.array([-30,-15,-8,-4,-2,0,2,4,8,15,30],np.float32)
DTW_OFFS =np.array([-20,-10,-5,-2,0,2,5,10,20],np.float32)

BEAMS=[(10,20.0,144.0,2,"cons"),(10,8.0,64.0,2,"loose"),(8,35.0,220.0,1,"vcons"),
       (10,14.0,90.0,5,"sm5"),(20,4.0,36.0,3,"vloose"),(12,12.0,100.0,3,"mid"),
       (15,25.0,180.0,2,"stiff")]

PF_N=600; ANCC_N=600; PF_N_SEEDS=5
PF_MOM=0.993; PF_VN=0.005; PF_PN=0.01; PF_IS=0.5; PF_RESAMP=0.5
PF_RP=0.2; PF_RV=0.003; PF_GW=5; PF_GWT=0.3
ANCC_A=0.998; ANCC_RN=0.002; ANCC_PN=0.005
ANCC_IS=0.3; ANCC_RP=0.1; ANCC_RR=0.001
PF_GS_MIN=10.; PF_GS_MAX=60.; PF_GS_DEF=30.
DTW_RADII=(20,50,100,200); DTW_STOCH_K=20; DTW_STOCH_T=3.0

# ── Numba JIT (identical to training notebook) ─────────────────────────────────
@njit(cache=True)
def _interp1(grid,v,vmin,step):
    i=int((v-vmin)/step)
    if i<0: return grid[0]
    n=len(grid)-1
    if i>=n: return grid[n]
    t=(v-vmin)/step-i; return grid[i]*(1.-t)+grid[i+1]*t

@njit(cache=True)
def _resamp(pos,aux,w,N,rp,rv):
    cum=np.zeros(N+1)
    for j in range(N): cum[j+1]=cum[j]+w[j]
    u0=np.random.uniform(0.,1./N); np2=np.empty(N); na=np.empty(N); ci=0
    for j in range(N):
        u=u0+j/N
        while ci<N-1 and cum[ci+1]<u: ci+=1
        np2[j]=pos[ci]+rp*np.random.randn(); na[j]=aux[ci]+rv*np.random.randn()
    return np2,na

@njit(cache=True)
def _beam_jit(sgr,tw_gr,si,BS,mc,es):
    n=len(sgr); nt=len(tw_gr); MAX=BS*6
    bidx=np.zeros(BS,np.int64); bidx[0]=si; bcost=np.full(BS,1e30); bcost[0]=0.; bn=np.int64(1)
    hI=np.zeros((n,BS),np.int64); hP=np.zeros((n,BS),np.int64)
    cI=np.zeros(MAX,np.int64); cC=np.full(MAX,1e30); cP=np.zeros(MAX,np.int64)
    for step in range(n):
        gv=sgr[step]; nc=np.int64(0)
        for bi in range(bn):
            idx=bidx[bi]; cost=bcost[bi]
            for d in range(-2,3):
                ni=idx+d
                if ni<0 or ni>=nt: continue
                tot=cost+(gv-tw_gr[ni])**2/es+mc*(d if d>=0 else -d)
                fnd=np.int64(-1)
                for ci in range(nc):
                    if cI[ci]==ni: fnd=ci; break
                if fnd>=0:
                    if tot<cC[fnd]: cC[fnd]=tot; cP[fnd]=bi
                else:
                    if nc<MAX: cI[nc]=ni; cC[nc]=tot; cP[nc]=bi; nc+=1
        kept=min(BS,nc)
        for i in range(kept):
            mi=i
            for j in range(i+1,nc):
                if cC[j]<cC[mi]: mi=j
            if mi!=i:
                cI[i],cI[mi]=cI[mi],cI[i]; cC[i],cC[mi]=cC[mi],cC[i]; cP[i],cP[mi]=cP[mi],cP[i]
        hI[step,:kept]=cI[:kept]; hP[step,:kept]=cP[:kept]
        bidx[:kept]=cI[:kept]; bcost[:kept]=cC[:kept]; bn=kept
    best=np.int64(0)
    for b in range(1,bn):
        if bcost[b]<bcost[best]: best=b
    path=np.zeros(n,np.int64); b=best
    for s in range(n-1,-1,-1): path[s]=hI[s,b]; b=hP[s,b]
    return path

@njit(cache=True)
def _pf_ancc_jit(md_v,z_v,gr_v,gg,vmin,step,gs,ls,ir,N,ALPHA,RN,PN,IS,RP,RR,RESAMP):
    pos=np.empty(N); rate=np.empty(N); w=np.ones(N)/N
    for j in range(N):
        pos[j]=ls+IS*np.random.randn(); rate[j]=ir+0.01*np.random.randn()
    pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.
    for i in range(len(md_v)):
        dm=max(md_v[i]-pm,1.)
        for j in range(N):
            rate[j]=ALPHA*rate[j]+RN*np.random.randn()
            pos[j]+=rate[j]*dm+PN*np.random.randn()
            tv=pos[j]-z_v[i]; tv=max(tv,vmin-50.); tv=min(tv,vmin+len(gg)*step+50.); pos[j]=tv+z_v[i]
        if not np.isnan(gr_v[i]):
            ws=0.
            for j in range(N):
                eg=_interp1(gg,pos[j]-z_v[i],vmin,step)
                d=(gr_v[i]-eg)/gs; lk=max(np.exp(-0.5*d*d) if d*d<600. else 0.,1e-300)
                w[j]*=lk; ws+=w[j]
            if ws>0.:
                for j in range(N): w[j]/=ws
            else:
                for j in range(N): w[j]=1./N
        ne=0.
        for j in range(N): ne+=w[j]*w[j]
        if 1./ne<RESAMP*N:
            pos,rate=_resamp(pos,rate,w,N,RP,RR)
            for j in range(N): w[j]=1./N
        tv=0.
        for j in range(N): tv+=w[j]*(pos[j]-z_v[i])
        pts[i]=tv; va=0.
        for j in range(N): va+=w[j]*(pos[j]-z_v[i]-tv)**2
        std_[i]=va**0.5; pm=md_v[i]
    return pts,std_

@njit(cache=True)
def _pf_z_jit(md_v,z_v,gr_v,gr_sm_v,gg_p,gg_s,vmin,step,gs,ip,iv,beta,icpt,zsig,N,
               MOM,VN,PN,GR_WT,RP,RV,RESAMP):
    pos=np.empty(N); vel=np.empty(N); w=np.ones(N)/N
    for j in range(N):
        pos[j]=ip+0.5*np.random.randn(); vel[j]=iv+0.02*np.random.randn()
    pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.; pz=z_v[0]-1.
    for i in range(len(md_v)):
        dm=max(md_v[i]-pm,1.); dzd=(z_v[i]-pz)/dm; ve=beta*dzd+icpt
        for j in range(N):
            vel[j]=MOM*vel[j]+VN*np.random.randn()
            pos[j]+=vel[j]*dm+PN*np.random.randn()
            pos[j]=max(pos[j],vmin-50.); pos[j]=min(pos[j],vmin+len(gg_p)*step+50.)
        if not np.isnan(gr_v[i]):
            ws=0.
            for j in range(N):
                ep=_interp1(gg_p,pos[j],vmin,step); dp=(gr_v[i]-ep)/gs
                lp=max(np.exp(-0.5*dp*dp) if dp*dp<600. else 0.,1e-300)
                if not np.isnan(gr_sm_v[i]):
                    es2=_interp1(gg_s,pos[j],vmin,step); ds=(gr_sm_v[i]-es2)/(gs*1.5)
                    ls_=max(np.exp(-0.5*ds*ds) if ds*ds<600. else 0.,1e-300)
                    lk=(1.-GR_WT)*lp+GR_WT*ls_
                else: lk=lp
                lk=max(lk,1e-300); w[j]*=lk; ws+=w[j]
            if ws>0.:
                for j in range(N): w[j]/=ws
            else:
                for j in range(N): w[j]=1./N
        ws2=0.
        for j in range(N):
            dv=(vel[j]-ve)/max(zsig*2.,0.005); lz=max(np.exp(-0.5*dv*dv) if dv*dv<600. else 0.,1e-300)
            w[j]*=lz; ws2+=w[j]
        if ws2>0.:
            for j in range(N): w[j]/=ws2
        else:
            for j in range(N): w[j]=1./N
        ne=0.
        for j in range(N): ne+=w[j]*w[j]
        if 1./ne<RESAMP*N:
            pos,vel=_resamp(pos,vel,w,N,RP,RV)
            for j in range(N): w[j]=1./N
        wm=0.
        for j in range(N): wm+=w[j]*pos[j]
        pts[i]=wm; va=0.
        for j in range(N): va+=w[j]*(pos[j]-wm)**2
        std_[i]=va**0.5; pm=md_v[i]; pz=z_v[i]
    return pts,std_

@njit(cache=True)
def _dtw_sc(query,ref,radius):
    N=len(query); M=len(ref); INF=1e18
    D=np.full((N,M),INF); slope=(M-1.0)/max(N-1.0,1.0)
    for i in range(N):
        jc=int(round(i*slope)); jlo=max(0,jc-radius); jhi=min(M-1,jc+radius)
        for j in range(jlo,jhi+1):
            cost=(query[i]-ref[j])**2
            if i==0 and j==0: D[i,j]=cost
            elif i==0: pv=D[i,j-1]; D[i,j]=cost+(pv if pv<INF else INF)
            elif j==0: pv=D[i-1,j]; D[i,j]=cost+(pv if pv<INF else INF)
            else:
                a=D[i-1,j-1]; b=D[i-1,j]; c=D[i,j-1]
                mn=a if a<b else b; mn=mn if mn<c else c
                D[i,j]=cost+(mn if mn<INF else INF)
    i=N-1; j=M-1; pi=np.zeros(N+M,np.int64); pj=np.zeros(N+M,np.int64); k=0
    while i>0 or j>0:
        pi[k]=i; pj[k]=j; k+=1
        if i==0: j-=1
        elif j==0: i-=1
        else:
            a=D[i-1,j-1]; b=D[i-1,j]; c=D[i,j-1]
            if a<=b and a<=c: i-=1; j-=1
            elif b<=c: i-=1
            else: j-=1
    pi[k]=0; pj[k]=0; k+=1
    return D,pi[:k],pj[:k]

@njit(cache=True)
def _dtw_path_to_tvt(pi,pj,tw_tvt,N):
    j4i=np.zeros(N,np.int64)
    for k in range(len(pi)): j4i[pi[k]]=pj[k]
    result=np.empty(N,np.float32)
    for i in range(N): result[i]=tw_tvt[j4i[i]]
    return result

@njit(cache=True)
def _dtw_path_slope(pi,pj,N,smooth_win=5):
    j4i=np.zeros(N,np.float64)
    for k in range(len(pi)): j4i[pi[k]]=float(pj[k])
    slope=np.zeros(N,np.float32); hw=smooth_win//2
    for i in range(N):
        i0=max(0,i-hw); i1=min(N-1,i+hw)
        slope[i]=float((j4i[i1]-j4i[i0])/(i1-i0)) if i1>i0 else 1.0
    return slope

@njit(cache=True)
def _dtw_stochastic(query,ref,radius,K,temperature):
    N=len(query); M=len(ref); INF=1e18; slope=(M-1.0)/max(N-1.0,1.0)
    D_base=np.full((N,M),INF)
    for i in range(N):
        jc=int(round(i*slope)); jlo=max(0,jc-radius); jhi=min(M-1,jc+radius)
        for j in range(jlo,jhi+1): D_base[i,j]=(query[i]-ref[j])**2
    paths=np.zeros((K,N),np.int64)
    for k in range(K):
        D=np.full((N,M),INF)
        for i in range(N):
            jc=int(round(i*slope)); jlo=max(0,jc-radius); jhi=min(M-1,jc+radius)
            for j in range(jlo,jhi+1):
                noise=-temperature*np.log(-np.log(np.random.uniform(1e-10,1.0)))
                cost=D_base[i,j]+noise
                if i==0 and j==0: D[i,j]=cost
                elif i==0: pv=D[i,j-1]; D[i,j]=cost+(pv if pv<INF else INF)
                elif j==0: pv=D[i-1,j]; D[i,j]=cost+(pv if pv<INF else INF)
                else:
                    a=D[i-1,j-1]; b=D[i-1,j]; c=D[i,j-1]
                    mn=a if a<b else b; mn=mn if mn<c else c
                    D[i,j]=cost+(mn if mn<INF else INF)
        i2=N-1; j2=M-1; j4i=np.zeros(N,np.int64)
        while i2>0 or j2>0:
            j4i[i2]=j2
            if i2==0: j2-=1
            elif j2==0: i2-=1
            else:
                a=D[i2-1,j2-1]; b=D[i2-1,j2]; c=D[i2,j2-1]
                if a<=b and a<=c: i2-=1; j2-=1
                elif b<=c: i2-=1
                else: j2-=1
        j4i[0]=j2; paths[k]=j4i
    return paths

def _make_grid(tw_tvt,tw_gr,step=0.2):
    tmin=float(tw_tvt.min()); tmax=float(tw_tvt.max())
    g=np.arange(tmin,tmax+step,step)
    return np.interp(g,tw_tvt,tw_gr).astype(np.float64),tmin,step

print("Compiling Numba JIT...")
_dg=np.ones(10,np.float64); _dw=np.ones(20,np.float64)
_dgg,_dm,_ds=_make_grid(np.arange(20,dtype=np.float64),_dw)
_dgg2,_,_=_make_grid(np.arange(20,dtype=np.float64),_dw+1.)
_beam_jit(_dg,_dw,5,5,10.,100.); _beam_jit(_dg,_dw,5,5,10.,100.)
_pf_ancc_jit(np.ones(5),np.zeros(5),np.ones(5),_dgg,_dm,_ds,30.,0.,0.,10,
             ANCC_A,ANCC_RN,ANCC_PN,ANCC_IS,ANCC_RP,ANCC_RR,0.5)
_pf_z_jit(np.ones(5,np.float64),np.zeros(5,np.float64),np.ones(5,np.float64),
          np.ones(5,np.float64),_dgg,_dgg2,_dm,_ds,30.,0.,0.,-1.,0.,0.1,10,
          0.993,0.005,0.01,0.3,0.2,0.003,0.5)
_q=np.random.randn(40); _r=np.random.randn(50)
_dtw_sc(_q,_r,10); _dtw_stochastic(_q,_r,10,3,2.0)
print("Numba JIT ready ✓")

# ── Signal helpers ─────────────────────────────────────────────────────────────
def run_dtw_multiscale(full_gr,tw_tvt,tw_gr,radii=DTW_RADII):
    N=len(full_gr)
    qn=((full_gr-full_gr.mean())/(full_gr.std()+1e-6)).astype(np.float64)
    rn=((tw_gr-tw_gr.mean())/(tw_gr.std()+1e-6)).astype(np.float64)
    tw_f32=tw_tvt.astype(np.float32)
    dtw_tvts={}; dtw_slopes={}; dtw_costs={}; tvt_stack=[]; inv_sum=0.
    for rad in radii:
        D,pi,pj=_dtw_sc(qn,rn,rad); cost=float(D[N-1,len(rn)-1])/max(N+len(rn),1)
        pi_f=pi[::-1]; pj_f=pj[::-1]
        tv=_dtw_path_to_tvt(pi_f,pj_f,tw_f32,N); sl=_dtw_path_slope(pi_f,pj_f,N)
        dtw_tvts[rad]=tv; dtw_slopes[rad]=sl; dtw_costs[rad]=cost
        ic=1./(cost+1e-6); inv_sum+=ic; tvt_stack.append((tv,ic))
    w=np.array([ic/inv_sum for _,ic in tvt_stack],np.float32)
    return dtw_tvts,dtw_slopes,dtw_costs,(np.stack([t for t,_ in tvt_stack],1)*w[None,:]).sum(1).astype(np.float32)

def run_dtw_stochastic(full_gr,tw_tvt,tw_gr,radius=50,K=DTW_STOCH_K,T=DTW_STOCH_T):
    N=len(full_gr)
    qn=((full_gr-full_gr.mean())/(full_gr.std()+1e-6)).astype(np.float64)
    rn=((tw_gr-tw_gr.mean())/(tw_gr.std()+1e-6)).astype(np.float64)
    tw_f32=tw_tvt.astype(np.float32); paths=_dtw_stochastic(qn,rn,radius,K,T)
    tvt_r=np.empty((K,N),np.float32)
    for k in range(K):
        for i in range(N): tvt_r[k,i]=tw_f32[paths[k,i]]
    mean_=tvt_r.mean(0).astype(np.float32); std_=tvt_r.std(0).astype(np.float32)
    return mean_,std_,(std_/(np.abs(mean_)+1e-6)).astype(np.float32)

def run_dtw_anchored(eval_gr,tw_tvt,tw_gr,last_tvt,radii=DTW_RADII):
    N=len(eval_gr); si=max(0,int(np.searchsorted(tw_tvt,last_tvt))-5)
    tw_suf_tvt=tw_tvt[si:]; tw_suf_gr=tw_gr[si:]
    if len(tw_suf_tvt)<5 or N<5: return np.full(N,last_tvt,np.float32),np.ones(N,np.float32)
    qn=((eval_gr-eval_gr.mean())/(eval_gr.std()+1e-6)).astype(np.float64)
    rn=((tw_suf_gr-tw_suf_gr.mean())/(tw_suf_gr.std()+1e-6)).astype(np.float64)
    tw_suf_f32=tw_suf_tvt.astype(np.float32); tvt_stack=[]; slope_stack=[]; inv_sum=0.
    for rad in radii:
        D,pi,pj=_dtw_sc(qn,rn,rad); cost=float(D[N-1,len(rn)-1])/max(N+len(rn),1)
        pi_f=pi[::-1]; pj_f=pj[::-1]
        tv=_dtw_path_to_tvt(pi_f,pj_f,tw_suf_f32,N); sl=_dtw_path_slope(pi_f,pj_f,N)
        ic=1./(cost+1e-6); inv_sum+=ic; tvt_stack.append((tv,ic)); slope_stack.append(sl)
    w=np.array([ic/inv_sum for _,ic in tvt_stack],np.float32)
    return (np.stack([t for t,_ in tvt_stack],1)*w[None,:]).sum(1).astype(np.float32), \
           np.stack(slope_stack,1).mean(1).astype(np.float32)

def compute_dwt_features(gr_full_vals,ev_start,nh,wavelet='db4',level=5):
    n=len(gr_full_vals)
    try:
        coeffs=pywt.wavedec(gr_full_vals.astype(np.float64),wavelet,level=level)
        approx_full=pywt.waverec([coeffs[0]]+[np.zeros_like(c) for c in coeffs[1:]],wavelet)[:n]
        detail_full=pywt.waverec([np.zeros_like(coeffs[0])]+[coeffs[1]]+[np.zeros_like(c) for c in coeffs[2:]],wavelet)[:n]
    except:
        approx_full=gr_full_vals.copy(); detail_full=np.zeros(n,np.float32)
    approx_ev=approx_full[ev_start:ev_start+nh].astype(np.float32)
    detail_ev=detail_full[ev_start:ev_start+nh].astype(np.float32)
    gr_ev=gr_full_vals[ev_start:ev_start+nh].astype(np.float32)
    return approx_ev,(detail_ev**2).astype(np.float32),(gr_ev-approx_ev).astype(np.float32)

def run_pf_ancc_multi(hw,tw_tvt,tw_gr,n_runs=PF_N_SEEDS):
    gg,vmin,step=_make_grid(tw_tvt,tw_gr)
    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0: return np.array([]),np.array([])
    k2=kn[kn['GR'].notna()]
    gs=PF_GS_DEF
    if len(k2)>=20:
        gs=float(np.clip(np.std(k2['GR'].values-np.interp(k2['TVT_input'].values,tw_tvt,tw_gr)),PF_GS_MIN,PF_GS_MAX))
    ls=float(kn['TVT_input'].iloc[-1])+float(kn['Z'].iloc[-1]); t=kn.tail(30); ir=0.
    if len(t)>=10:
        dt=np.diff(t['TVT_input'].values); dz=np.diff(t['Z'].values); dm=np.diff(t['MD'].values); m=dm>0
        if m.sum()>=3: ir=float(np.median((dt[m]+dz[m])/dm[m]))
    md_v=ev['MD'].to_numpy(np.float64); z_v=ev['Z'].to_numpy(np.float64); gr_v=ev['GR'].to_numpy(np.float64)
    all_pts=[]
    for _ in range(n_runs):
        pts,_=_pf_ancc_jit(md_v,z_v,gr_v,gg,vmin,step,gs,ls,ir,ANCC_N,ANCC_A,ANCC_RN,ANCC_PN,ANCC_IS,ANCC_RP,ANCC_RR,PF_RESAMP)
        all_pts.append(pts)
    stk=np.stack(all_pts,0); return stk.mean(0).astype(np.float32),stk.std(0).astype(np.float32)

def run_pf_z_multi(hw,tw_tvt,tw_gr,n_runs=PF_N_SEEDS):
    gg_p,vmin,step=_make_grid(tw_tvt,tw_gr)
    tw_sm=pd.Series(tw_gr).rolling(PF_GW,center=True,min_periods=1).mean().values
    gg_s,_,_=_make_grid(tw_tvt,tw_sm)
    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0: return np.array([]),np.array([])
    k2=kn[kn['GR'].notna()]
    gs=PF_GS_DEF
    if len(k2)>=20:
        gs=float(np.clip(np.std(k2['GR'].values-np.interp(k2['TVT_input'].values,tw_tvt,tw_gr)),PF_GS_MIN,PF_GS_MAX))
    ktvt=kn['TVT_input'].values; kmd=kn['MD'].values; kz=kn['Z'].values
    dz=np.diff(kz); dtvt=np.diff(ktvt); dmd_=np.diff(kmd); m=dmd_>0
    beta,intc,zsig=-1.,0.,0.1
    if m.sum()>=10:
        vz=dz[m]/dmd_[m]; vt=dtvt[m]/dmd_[m]
        c,_,_,_=np.linalg.lstsq(np.column_stack([vz,np.ones_like(vz)]),vt,rcond=None)
        beta,intc=float(c[0]),float(c[1]); zsig=max(float(np.std(vt-(c[0]*vz+c[1]))),0.001)
    iv=0.
    if len(kn)>=10:
        t2=kn.tail(20); dt2=np.diff(t2['TVT_input'].values); dm2=np.diff(t2['MD'].values); m2=dm2>0
        if m2.sum()>=3: iv=float(np.median(dt2[m2]/dm2[m2]))
    ip=float(kn['TVT_input'].iloc[-1])
    gr_full=hw['GR'].astype(float).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
    hw_gsm=gr_full.rolling(PF_GW,center=True,min_periods=1).mean()
    gr_v=ev['GR'].to_numpy(np.float64); gr_sm_v=hw_gsm.iloc[ev.index].to_numpy(np.float64)
    md_v=ev['MD'].to_numpy(np.float64); z_v=ev['Z'].to_numpy(np.float64); all_pts=[]
    for _ in range(n_runs):
        pts,_=_pf_z_jit(md_v,z_v,gr_v,gr_sm_v,gg_p,gg_s,vmin,step,gs,ip,iv,beta,intc,zsig,PF_N,PF_MOM,PF_VN,PF_PN,PF_GWT,PF_RP,PF_RV,PF_RESAMP)
        all_pts.append(pts)
    stk=np.stack(all_pts,0); return stk.mean(0).astype(np.float32),stk.std(0).astype(np.float32)

# ── Imputer classes — full implementations, build from competition data ─────────
# These are identical to rogii_v13_train.py.
# Building takes ~1 min; avoids all pickle/AttributeError issues entirely.

class FormationPlaneKNN:
    def __init__(self, wids, data_dir):
        rows=[]
        for wid in wids:
            p=data_dir/f'{wid}__horizontal_well.csv'
            try: df=pd.read_csv(p,usecols=['X','Y']+FORMATIONS).dropna()
            except: continue
            if len(df)==0: continue
            row={'wid':wid,'x':float(df['X'].median()),'y':float(df['Y'].median())}
            for c in FORMATIONS: row[f'{c}_m']=float(df[c].median())
            rows.append(row)
        self.df=pd.DataFrame(rows); self.wmap={w:i for i,w in enumerate(self.df['wid'])}
        xy=self.df[['x','y']].to_numpy(); self.scale=np.where(xy.std(0)<1e-3,1.,xy.std(0))
        self.tree=cKDTree(xy/self.scale); self.xa=self.df['x'].to_numpy()
        self.ya=self.df['y'].to_numpy()
        self.fa=self.df[[f'{c}_m' for c in FORMATIONS]].to_numpy(np.float64)
    def impute(self,xy_q,self_wid=None,k=PLANE_K):
        q=xy_q/self.scale; nf=min(k+5,len(self.df)); dist,idx=self.tree.query(q,k=nf,workers=-1)
        if self_wid in self.wmap: dist=np.where(idx==self.wmap[self_wid],np.inf,dist)
        ord_=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
        dk=np.take_along_axis(dist,ord_,1); ik=np.take_along_axis(idx,ord_,1)
        vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.).astype(np.float64)
        xn=self.xa[ik]; yn=self.ya[ik]; fn=self.fa[ik]; wx=w*xn; wy=w*yn
        A=np.zeros((len(q),3,3))
        A[:,0,0]=(wx*xn).sum(1); A[:,0,1]=(wx*yn).sum(1); A[:,0,2]=wx.sum(1)
        A[:,1,0]=A[:,0,1]; A[:,1,1]=(wy*yn).sum(1); A[:,1,2]=wy.sum(1)
        A[:,2,0]=A[:,0,2]; A[:,2,1]=A[:,1,2]; A[:,2,2]=w.sum(1)
        A[:,0,0]+=1e-9; A[:,1,1]+=1e-9; A[:,2,2]+=1e-9
        rhs=np.stack([(wx[:,:,None]*fn).sum(1),(wy[:,:,None]*fn).sum(1),(w[:,:,None]*fn).sum(1)],1)
        try: coef=np.linalg.solve(A,rhs)
        except:
            coef=np.zeros((len(q),3,6))
            for r2 in range(len(q)):
                try: coef[r2]=np.linalg.pinv(A[r2])@rhs[r2]
                except: pass
        Xq=xy_q[:,0]; Yq=xy_q[:,1]
        pred=(Xq[:,None]*coef[:,0,:]+Yq[:,None]*coef[:,1,:]+coef[:,2,:]).astype(np.float32)
        pred[~vk.any(1)]=self.fa.mean(0)
        return pred,np.where(vk,dk,np.inf).min(1).astype(np.float32)
    def get_neighbor_wids(self,xy_q,self_wid=None,k=2):
        q=np.atleast_2d(xy_q)/self.scale; nf=min(k+10,len(self.df))
        dist,idx=self.tree.query(q,k=nf,workers=-1); dist=dist[0]; idx=idx[0]
        if self_wid in self.wmap: dist=np.where(idx==self.wmap[self_wid],np.inf,dist)
        order=np.argsort(dist); wids_arr=self.df['wid'].values; out=[]
        for i in order:
            if np.isfinite(dist[i]): out.append((wids_arr[idx[i]],float(dist[i])))
            if len(out)==k: break
        return out

class DenseANCCImputer:
    def __init__(self,wids,data_dir,spw=DENSE_SPW):
        xs,ys,anccs,wids_=[],[],[],[]
        for wid in wids:
            p=data_dir/f'{wid}__horizontal_well.csv'
            try: df=pd.read_csv(p,usecols=['X','Y','ANCC']).dropna()
            except: continue
            if len(df)==0: continue
            ix=np.linspace(0,len(df)-1,min(spw,len(df)),dtype=int); s=df.iloc[ix]
            xs.append(s['X'].values); ys.append(s['Y'].values)
            anccs.append(s['ANCC'].values); wids_.extend([wid]*len(s))
        self.xy=np.column_stack([np.concatenate(xs),np.concatenate(ys)])
        self.ancc=np.concatenate(anccs).astype(np.float32); self.wids=np.array(wids_)
        self.scale=np.where(self.xy.std(0)<1e-3,1.,self.xy.std(0))
        self.tree=cKDTree(self.xy/self.scale)
    def impute(self,xy_q,self_wid=None,k=DENSE_K,nfetch=5000):
        xy_q=np.atleast_2d(xy_q); q=xy_q/self.scale; nf=min(nfetch,len(self.ancc))
        dist,idx=self.tree.query(q,k=nf,workers=-1)
        if self_wid: dist=np.where(self.wids[idx]==self_wid,np.inf,dist)
        ord_=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
        dk=np.take_along_axis(dist,ord_,1); ik=np.take_along_axis(idx,ord_,1)
        vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.)
        sw=w.sum(1); safe=np.where(sw<1e-9,1.,sw); an=self.ancc[ik]
        ap=(an*w).sum(1)/safe; ap=np.where(sw<1e-9,float(self.ancc.mean()),ap)
        var=((an-ap[:,None])**2*w).sum(1)/safe
        return (ap.astype(np.float32),np.sqrt(np.maximum(var,0.)).astype(np.float32),
                np.where(vk,dk,np.inf).min(1).astype(np.float32))

class TypewellCache:
    def __init__(self,wids,data_dir):
        self.cache={}
        for wid in wids:
            p=data_dir/f'{wid}__typewell.csv'
            try:
                tw=pd.read_csv(p).sort_values('TVT').dropna(subset=['TVT','GR'])
                if len(tw)>=5:
                    self.cache[wid]=(tw['TVT'].values.astype(np.float32),tw['GR'].values.astype(np.float32))
            except: pass
    def get(self,wid): return self.cache.get(wid,None)

# ── STEP 1: Build imputers from competition training data ──────────────────────
# The training data is always available in the competition input, so we build
# FI / DI / TW_CACHE fresh every run (~1 min).  No pickle files needed —
# completely avoids the AttributeError that occurs when loading class instances
# saved from a different Python module.

print("\nBuilding imputers from competition train/ data (~1 min)...")
t0_imp = time.time()

hw_paths_train = sorted(TRAIN_DIR.glob('*__horizontal_well.csv'))
train_wids     = [p.stem.replace('__horizontal_well','') for p in hw_paths_train]
print(f"  {len(train_wids)} training wells found")

FI       = FormationPlaneKNN(train_wids, TRAIN_DIR)
DI       = DenseANCCImputer(train_wids, TRAIN_DIR)
TW_CACHE = TypewellCache(train_wids, TRAIN_DIR)

print(f"  FPK:{len(FI.df)} | Dense:{len(DI.ancc):,} | TW:{len(TW_CACHE.cache)}")
print(f"  Imputers built in {time.time()-t0_imp:.0f}s ✓")

# Load model meta from artifacts (models/ and meta/ only — no imputers/ needed)
feature_cols = joblib.load(ARTS/"meta/feature_cols.pkl")
hc_meta      = joblib.load(ARTS/"meta/hc_weights.pkl")
pp_params    = joblib.load(ARTS/"meta/pp_params.pkl")
ALPHA=pp_params['alpha']; TAU=pp_params['tau']; W_PF=pp_params['w_pf']
print(f"  {len(feature_cols)} features")
print(f"  Ensemble: {hc_meta['method']}  |  PP: alpha={ALPHA} tau={TAU} w_pf={W_PF}")

# Sync module-level globals used inside build_well_test
_FI=FI; _DI=DI; _TW=TW_CACHE

# ── Helpers ────────────────────────────────────────────────────────────────────
def robust_slope(x,y):
    x=np.asarray(x,float); y=np.asarray(y,float); m=np.isfinite(x)&np.isfinite(y)
    if m.sum()<2 or np.std(x[m])<1e-6: return 0.
    return float(np.polyfit(x[m],y[m],1)[0])

def affine_cal(kgr,tw_at_k,min_pts=20):
    v=np.isfinite(kgr)&np.isfinite(tw_at_k)
    if v.sum()<min_pts or np.std(tw_at_k[v])<1e-6:
        return 1.,float(np.nanmean(kgr[v])-np.nanmean(tw_at_k[v])) if v.any() else 0.
    a,b=np.polyfit(tw_at_k[v],kgr[v],1); return float(a),float(b)

def seg_b_well(ktvt,kz,form_col):
    bv=ktvt+kz-form_col; n=len(bv); b_full=float(np.median(bv))
    b_late=float(np.median(bv[max(0,n-50):])) if n>=5 else b_full
    t1,t2=n//3,2*n//3
    b_early=float(np.median(bv[:max(1,t1)])) if t1>0 else b_full
    b_mid  =float(np.median(bv[t1:max(t1+1,t2)])) if t2>t1 else b_full
    w=np.exp(0.02*np.arange(n)); w/=w.sum(); b_wls=float(np.dot(w,bv))
    return b_full,b_early,b_mid,b_late,b_wls

def multi_scale_ncc(kgr,ktvt,hgr,hws=(8,15,25),stride=3):
    out=[]
    for hw_sc in hws:
        win=2*hw_sc+1; nk=len(kgr); nh=len(hgr)
        if nk<win+1 or nh==0: out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
        kg=pd.Series(kgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
        hg=pd.Series(hgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
        sts=np.arange(0,nk-win+1,stride,dtype=np.int32)
        if len(sts)==0: out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
        C=kg[sts[:,None]+np.arange(win,dtype=np.int32)[None,:]].astype(np.float32)
        Cn=(C-C.mean(1,keepdims=True))/(C.std(1,keepdims=True)+1e-6)
        hp=np.pad(hg,hw_sc,mode='edge'); H=hp[np.arange(nh)[:,None]+np.arange(win)[None,:]].astype(np.float32)
        Hn=(H-H.mean(1,keepdims=True))/(H.std(1,keepdims=True)+1e-6)
        ncc=Hn@Cn.T/win; best=ncc.argmax(1); score=ncc.max(1).astype(np.float32)
        out.append((ktvt[np.clip(sts[best]+hw_sc,0,nk-1)].astype(np.float32),score))
    tvts=np.stack([o[0] for o in out],1); scores=np.stack([o[1] for o in out],1)
    sw=np.exp(3.*scores); sw/=sw.sum(1,keepdims=True)+1e-9
    return out,(tvts*sw).sum(1).astype(np.float32)

def _beam_raw(hgr,tw_tvt,tw_gr,start_tvt,bs,mc,es,r):
    sgr=pd.Series(hgr.astype(np.float64)).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
    if r>0: sgr=sgr.rolling(r*2+1,center=True,min_periods=1).mean()
    sgr=sgr.to_numpy(np.float64); si=int(np.searchsorted(tw_tvt,start_tvt)); si=max(0,min(si,len(tw_tvt)-1))
    return tw_tvt[_beam_jit(sgr,tw_gr.astype(np.float64),si,bs,float(mc),float(es)).clip(0,len(tw_tvt)-1)].astype(np.float32)

# ── STEP 2: Build test features ────────────────────────────────────────────────
def build_well_test(hw_path,tw_path):
    """Feature builder for test wells — identical to training but is_train=False."""
    global _FI,_DI,_TW
    wid=Path(hw_path).stem.replace('__horizontal_well','')
    try: hw=pd.read_csv(hw_path); tw=pd.read_csv(tw_path).sort_values('TVT')
    except: return None
    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0 or len(kn)<10: return None
    tw_tvt=tw['TVT'].to_numpy(np.float32); tw_gr=tw['GR'].to_numpy(np.float32)
    if len(tw_tvt)<3: return None
    np.random.seed(SEED)
    pf_a_ms,pf_a_ms_std=run_pf_ancc_multi(hw,tw_tvt,tw_gr)
    if len(pf_a_ms)==0: return None
    pf_z_ms,pf_z_ms_std=run_pf_z_multi(hw,tw_tvt,tw_gr)
    pf_use=pf_a_ms.astype(np.float32); std_use=pf_a_ms_std.astype(np.float32)
    has_z=len(pf_z_ms)==len(pf_a_ms) and not np.any(np.isnan(pf_z_ms))
    lk=kn.iloc[-1]; last_tvt=float(lk['TVT_input'])
    gr_full=hw['GR'].astype(float).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
    hgr=gr_full.iloc[ev.index[0]:].to_numpy(np.float32)
    kgr=gr_full.iloc[:len(kn)].to_numpy(np.float32)
    ktvt=kn['TVT_input'].to_numpy(np.float32); full_gr_np=gr_full.values.astype(np.float32)
    bpaths={}
    for (bs,mc,es,r,tag) in BEAMS: bpaths[tag]=_beam_raw(hgr,tw_tvt,tw_gr,last_tvt,bs,mc,es,r)
    beam_ref=(bpaths['cons']+bpaths['sm5'])/2.
    end_tvt_est=float(pf_use[-1]) if len(pf_use)>0 else last_tvt+20.
    back_cons=_beam_raw(hgr[::-1].copy(),tw_tvt,tw_gr,end_tvt_est,BEAMS[0][0],BEAMS[0][1],BEAMS[0][2],BEAMS[0][3])[::-1].copy()
    back_loose=_beam_raw(hgr[::-1].copy(),tw_tvt,tw_gr,end_tvt_est,BEAMS[1][0],BEAMS[1][1],BEAMS[1][2],BEAMS[1][3])[::-1].copy()
    bidir_cons=(0.6*bpaths['cons']+0.4*back_cons).astype(np.float32)
    sc_res,sc_ens=multi_scale_ncc(kgr,ktvt,hgr)
    sc8,sc8s=sc_res[0]; sc15,sc15s=sc_res[1]; sc25,sc25s=sc_res[2]
    sc_cons=(sc8+sc15+sc25)/3.; sc_trust=float(np.clip(len(kn)/200.,0.,0.6))
    hyb_ref=(1-sc_trust)*beam_ref+sc_trust*sc_ens
    dtw_tvts,dtw_slopes,dtw_costs,dtw_ens_full=run_dtw_multiscale(full_gr_np,tw_tvt,tw_gr)
    dtw_mean_full,dtw_std_full,dtw_cv_full=run_dtw_stochastic(full_gr_np,tw_tvt,tw_gr)
    dtw_anch_ens,dtw_anch_slope=run_dtw_anchored(hgr,tw_tvt,tw_gr,last_tvt)
    nh=len(ev); ev_start=ev.index[0]
    def _ev(arr): return arr[ev_start:ev_start+nh].astype(np.float32)
    dtw_ens_ev=_ev(dtw_ens_full); dtw_mean_ev=_ev(dtw_mean_full)
    dtw_std_ev=_ev(dtw_std_full); dtw_cv_ev=_ev(dtw_cv_full)
    dtw_per_r={r:_ev(dtw_tvts[r]) for r in DTW_RADII}
    dtw_slope_r={r:_ev(dtw_slopes[r]) for r in DTW_RADII}
    dtw_slope_mean=np.stack([dtw_slope_r[r] for r in DTW_RADII],1).mean(1).astype(np.float32)
    dtw_cost_arr=np.array([dtw_costs[r] for r in DTW_RADII],np.float32)
    tw_at_k=np.interp(ktvt,tw_tvt,tw_gr).astype(np.float32)
    a_cal,b_cal=affine_cal(kgr,tw_at_k)
    kmd=kn['MD'].to_numpy(np.float32); kz=kn['Z'].to_numpy(np.float32)
    pfx_rmse=float(np.sqrt(np.mean((kgr-tw_at_k)**2)))
    slp_all=robust_slope(kmd,ktvt); slp_50=robust_slope(kmd[-50:],ktvt[-50:]); slp_z=robust_slope(kn['Z'].to_numpy(),ktvt)
    def _tail_slope(k):
        tail=kn.tail(k)
        if len(tail)<2: return slp_all
        return robust_slope(tail['MD'].to_numpy(),tail['TVT_input'].to_numpy())
    slp_k10=_tail_slope(10); slp_k25=_tail_slope(25); slp_k50=_tail_slope(50); slp_k100=_tail_slope(100)
    # Test wells: self_wid=None (no leave-one-out exclusion needed)
    xy_ev=ev[['X','Y']].to_numpy(np.float64); xy_kn=kn[['X','Y']].to_numpy(np.float64)
    form_ev,knn_d=_FI.impute(xy_ev,self_wid=None); form_kn,_=_FI.impute(xy_kn,self_wid=None)
    z_kn=kn['Z'].to_numpy(np.float32); z_ev=ev['Z'].to_numpy(np.float32)
    tvt_fs={}; form_rmse={}; form_list=[]
    for fi2,fn in enumerate(FORMATIONS):
        b_full,b_early,b_mid,b_late,b_wls=seg_b_well(ktvt,z_kn,form_kn[:,fi2])
        tvt_fs[f'tvtF_{fn}']    =(-z_ev+form_ev[:,fi2]+b_full).astype(np.float32)
        tvt_fs[f'tvtFw_{fn}']   =(-z_ev+form_ev[:,fi2]+b_wls ).astype(np.float32)
        tvt_fs[f'tvtF50_{fn}']  =(-z_ev+form_ev[:,fi2]+b_late).astype(np.float32)
        tvt_fs[f'bw_{fn}']=np.float32(b_full); tvt_fs[f'bww_{fn}']=np.float32(b_wls); tvt_fs[f'bw50_{fn}']=np.float32(b_late)
        tvt_fs[f'bw_early_{fn}']=np.float32(b_early); tvt_fs[f'bw_mid_{fn}']=np.float32(b_mid)
        form_rmse[fn]=float(np.sqrt(np.mean((ktvt-(-z_kn+form_kn[:,fi2]+b_full))**2))); form_list.append(tvt_fs[f'tvtF_{fn}'])
    fs=np.stack(form_list,1); form_mean_d=(fs.mean(1)-last_tvt).astype(np.float32)
    form_std_d=fs.std(1).astype(np.float32); form_rng_d=(fs.max(1)-fs.min(1)).astype(np.float32)
    d_ancc,d_std,d_dist=_DI.impute(xy_ev,self_wid=None); d_kn,_,_=_DI.impute(xy_kn,self_wid=None)
    b_vd=ktvt+z_kn-d_kn; _,_,_,b_dl,b_dw=seg_b_well(ktvt,z_kn,d_kn); b_d=float(np.median(b_vd))
    tvt_dense=(-z_ev+d_ancc+b_d).astype(np.float32); tvt_densew=(-z_ev+d_ancc+b_dw).astype(np.float32)
    tvt_dense50=(-z_ev+d_ancc+b_dl).astype(np.float32)
    d_rmse=float(np.sqrt(np.mean((ktvt+z_kn-d_kn-b_d)**2))); d_bias=float(np.mean(b_vd-b_d))
    center_xy=np.mean(xy_ev,axis=0,keepdims=True); ntw_feats={}
    neighbors=_FI.get_neighbor_wids(center_xy,self_wid=None,k=2)
    for ni in range(2):
        kd=f'ntw{ni+1}_beam_d'; kdi=f'ntw{ni+1}_dist'
        if ni<len(neighbors):
            n_wid,n_dist=neighbors[ni]; ntw_data=_TW.get(n_wid)
            if ntw_data is not None:
                ntw_tvt,ntw_gr=ntw_data
                ntw_b=_beam_raw(hgr,ntw_tvt,ntw_gr,last_tvt,BEAMS[0][0],BEAMS[0][1],BEAMS[0][2],BEAMS[0][3])
                ntw_feats[kd]=(ntw_b-last_tvt).astype(np.float32); ntw_feats[kdi]=np.full(nh,np.float32(n_dist))
            else: ntw_feats[kd]=np.zeros(nh,np.float32); ntw_feats[kdi]=np.full(nh,np.float32(1e6))
        else: ntw_feats[kd]=np.zeros(nh,np.float32); ntw_feats[kdi]=np.full(nh,np.float32(1e6))
    est_stack=np.stack([pf_use,bpaths['cons'],sc_ens,dtw_ens_ev,tvt_fs['tvtF_ANCC']],1)
    estimator_range=(est_stack.max(1)-est_stack.min(1)).astype(np.float32)
    estimator_std=est_stack.std(1).astype(np.float32)
    estimator_max_d=(est_stack.max(1)-last_tvt).astype(np.float32)
    estimator_min_d=(est_stack.min(1)-last_tvt).astype(np.float32)
    all_sigs=[pf_use]+list(bpaths.values())+[sc8,sc15,sc25,sc_ens,tvt_fs['tvtF_ANCC'],tvt_dense]
    sig_mat=np.stack(all_sigs,1); sig_std=sig_mat.std(1).astype(np.float32); sig_mean=(sig_mat.mean(1)-last_tvt).astype(np.float32)
    gr_s=pd.Series(gr_full.values); rolls={}
    for w in [5,21,51,101]:
        r_=gr_s.rolling(w,center=True,min_periods=1)
        rolls[f'grm{w}']=r_.mean().iloc[ev.index].values.astype(np.float32); rolls[f'grs{w}']=r_.std().fillna(0).iloc[ev.index].values.astype(np.float32)
    for lag in [1,5,15,30]:
        rolls[f'glag{lag}']=gr_s.shift(lag).bfill().iloc[ev.index].values.astype(np.float32)
        rolls[f'glead{lag}']=gr_s.shift(-lag).ffill().iloc[ev.index].values.astype(np.float32)
    gr_d1=gr_s.diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_d2=gr_s.diff().diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_env=gr_s.rolling(21,center=True,min_periods=1).max().iloc[ev.index].values.astype(np.float32)
    gr_nrg=np.sqrt(np.maximum((gr_s**2).rolling(21,center=True,min_periods=1).mean(),0.)).iloc[ev.index].values.astype(np.float32)
    hmd=ev['MD'].to_numpy(np.float32); md_since=hmd-float(lk['MD'])
    slp_b_all=(last_tvt+slp_all*md_since).astype(np.float32); slp_b_50=(last_tvt+slp_50*md_since).astype(np.float32)
    mdd=hw['MD'].diff().replace(0,np.nan)
    dzdmd=(hw['Z'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
    dxdmd=(hw['X'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
    dydmd=(hw['Y'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
    d2z=(hw['Z'].diff()/mdd).diff().fillna(0.); d2x=(hw['X'].diff()/mdd).diff().fillna(0.); d2y=(hw['Y'].diff()/mdd).diff().fillna(0.)
    dls=np.nan_to_num(np.sqrt(d2z**2+d2x**2+d2y**2).iloc[ev.index].values.astype(np.float32))
    gr_rank=gr_s.rank(pct=True).iloc[ev.index].values.astype(np.float32)
    gr_dwt_approx5,gr_dwt_detail_energy,gr_dwt_residual=compute_dwt_features(full_gr_np,ev_start,nh)
    frac=(np.arange(nh)/max(nh-1,1)).astype(np.float32)
    hgr_fill=gr_full.iloc[ev.index].values.astype(np.float32)
    def sc(v): return np.full(nh,np.float32(v),np.float32)
    feats={
        'well':wid,'id':[f'{wid}_{i}' for i in ev.index],'last_known_tvt':sc(last_tvt),
        'pf_ancc':pf_use,'pf_ancc_std':std_use,'pf_ancc_delta':(pf_use-last_tvt).astype(np.float32),
        'pf_ancc_ms_std':pf_a_ms_std,
        'pf_z':(pf_z_ms.astype(np.float32) if has_z else sc(last_tvt)),
        'pf_z_delta':((pf_z_ms-last_tvt).astype(np.float32) if has_z else sc(0.)),
        'pf_z_ms_std':(pf_z_ms_std.astype(np.float32) if has_z else sc(0.)),
        'pf_vs_z':((pf_use-pf_z_ms.astype(np.float32)) if has_z else sc(0.)),
        **{f'beam_{t}_d':(p-np.float32(last_tvt)).astype(np.float32) for t,p in bpaths.items()},
        'beam_mean_d':np.stack([(p-last_tvt) for p in bpaths.values()],1).mean(1).astype(np.float32),
        'beam_std_d': np.stack([(p-last_tvt) for p in bpaths.values()],1).std(1).astype(np.float32),
        'beam_med_d': np.median(np.stack([(p-last_tvt) for p in bpaths.values()],1),1).astype(np.float32),
        'beam_back_cons_d':(back_cons-last_tvt).astype(np.float32),'beam_back_loose_d':(back_loose-last_tvt).astype(np.float32),
        'beam_bidir_cons_d':(bidir_cons-last_tvt).astype(np.float32),'fwd_vs_back_cons':(bpaths['cons']-back_cons).astype(np.float32),
        'sc8_d':(sc8-np.float32(last_tvt)).astype(np.float32),'sc8_sc':sc8s,
        'sc15_d':(sc15-np.float32(last_tvt)).astype(np.float32),'sc15_sc':sc15s,
        'sc25_d':(sc25-np.float32(last_tvt)).astype(np.float32),'sc25_sc':sc25s,
        'sc_cons_d':(sc_cons-np.float32(last_tvt)).astype(np.float32),'sc_ens_d':(sc_ens-np.float32(last_tvt)).astype(np.float32),
        'sc_trust':sc(sc_trust),'hyb_d':(hyb_ref-np.float32(last_tvt)).astype(np.float32),
        'dtw_ens_d':(dtw_ens_ev-last_tvt).astype(np.float32),
        **{f'dtw_r{r}_d':(dtw_per_r[r]-last_tvt).astype(np.float32) for r in DTW_RADII},
        **{f'dtw_slope_r{r}':dtw_slope_r[r] for r in DTW_RADII},
        'dtw_slope_mean':dtw_slope_mean,
        'dtw_cost_min':sc(float(dtw_cost_arr.min())),'dtw_cost_range':sc(float(dtw_cost_arr.max()-dtw_cost_arr.min())),
        'dtw_stoch_mean_d':(dtw_mean_ev-last_tvt).astype(np.float32),'dtw_stoch_std':dtw_std_ev,'dtw_stoch_cv':dtw_cv_ev,
        'dtw_vs_beam':(dtw_ens_ev-bpaths['cons']).astype(np.float32),
        'dtw_vs_pf':(dtw_ens_ev-pf_use).astype(np.float32),'dtw_vs_sc':(dtw_ens_ev-sc_ens).astype(np.float32),
        'dtw_anch_d':(dtw_anch_ens-last_tvt).astype(np.float32),'dtw_anch_slope':dtw_anch_slope,
        'dtw_anch_vs_full':(dtw_anch_ens-dtw_ens_ev).astype(np.float32),
        'gr_dwt_approx5':gr_dwt_approx5,'gr_dwt_detail_energy':gr_dwt_detail_energy,'gr_dwt_residual':gr_dwt_residual,
        'estimator_range':estimator_range,'estimator_std':estimator_std,'estimator_max_d':estimator_max_d,'estimator_min_d':estimator_min_d,
        'slp_k10':sc(slp_k10),'slp_k25':sc(slp_k25),'slp_k50':sc(slp_k50),'slp_k100':sc(slp_k100),
        'slope_accel_10_50':sc(slp_k10-slp_k50),'slope_accel_25_100':sc(slp_k25-slp_k100),
        'sig_std':sig_std,'sig_mean_d':sig_mean,
        **tvt_fs,**{f'frm_rmse_{fn}':sc(form_rmse[fn]) for fn in FORMATIONS},
        'form_mean_d':form_mean_d,'form_std_d':form_std_d,'form_rng_d':form_rng_d,
        'spatial_knn_dist':knn_d,'dense_ancc':d_ancc,'dense_std':d_std,'dense_dist':d_dist,
        'tvt_dense_d':(tvt_dense-last_tvt).astype(np.float32),'tvt_densew_d':(tvt_densew-last_tvt).astype(np.float32),
        'tvt_dense50_d':(tvt_dense50-last_tvt).astype(np.float32),'dense_rmse':sc(d_rmse),'dense_bias':sc(d_bias),
        'pf_vs_spatial':(pf_use-tvt_fs['tvtF_ANCC']).astype(np.float32),
        'pf_vs_dense':(pf_use-tvt_dense).astype(np.float32),
        'spatial_vs_dense':(tvt_fs['tvtF_ANCC']-tvt_dense).astype(np.float32),
        'beam_vs_spatial':(bpaths['cons']-tvt_fs['tvtF_ANCC']).astype(np.float32),
        'sc_vs_beam':(sc_ens-bpaths['cons']).astype(np.float32),'bidir_vs_fwd':(bidir_cons-bpaths['cons']).astype(np.float32),
        **ntw_feats,
        'cal_a':sc(a_cal),'cal_b':sc(b_cal),'pfx_rmse':sc(pfx_rmse),
        'known_len':sc(len(kn)),'eval_len':sc(nh),
        'slp_all':sc(slp_all),'slp_50':sc(slp_50),'slp_z':sc(slp_z),
        'slp_b_d_all':(slp_b_all-last_tvt).astype(np.float32),'slp_b_d_50':(slp_b_50-last_tvt).astype(np.float32),
        'dzdmd':dzdmd,'dxdmd':dxdmd,'dydmd':dydmd,'dls':dls,'gr_rank':gr_rank,
        'md_since':md_since,'frac':frac,'frac2':frac**2,'ease_frac':(3*frac**2-2*frac**3).astype(np.float32),
        'z':z_ev,'x':ev['X'].to_numpy(np.float32),'y':ev['Y'].to_numpy(np.float32),
        **rolls,'gr_d1':gr_d1,'gr_d2':gr_d2,'gr_env':gr_env,'gr_nrg':gr_nrg,
        'gr_minus_tw_last':(hgr_fill-float(np.interp(last_tvt,tw_tvt,tw_gr))).astype(np.float32),
        'anchor_t_pos':sc(float((last_tvt-float(tw_tvt.min()))/max(float(tw_tvt.max()-tw_tvt.min()),1e-3))),
        'tw_tvt_range':sc(float(tw_tvt.max()-tw_tvt.min())),'tw_gr_mean':sc(float(tw_gr.mean())),'tw_gr_std':sc(float(tw_gr.std())),
    }
    for o in ANCH_OFFS: feats[f'anch_diff_{int(o)}']=hgr_fill-float(np.interp(last_tvt+float(o),tw_tvt,tw_gr))
    for o in BEAM_OFFS: feats[f'beam_diff_{int(o)}']=hgr_fill-np.interp(bpaths['cons']+float(o),tw_tvt,tw_gr).astype(np.float32)
    for o in SC_OFFS:   feats[f'sc_diff_{int(o)}']=hgr_fill-np.interp(sc_ens+float(o),tw_tvt,tw_gr).astype(np.float32)
    for o in PF_OFFS:   feats[f'pf_diff_{int(o)}']=hgr_fill-np.interp(pf_use+float(o),tw_tvt,tw_gr).astype(np.float32)
    for o in DTW_OFFS:  feats[f'dtw_diff_{int(o)}']=hgr_fill-np.interp(dtw_ens_ev+float(o),tw_tvt,tw_gr).astype(np.float32)
    df=pd.DataFrame(feats)
    for c in df.select_dtypes('float64').columns: df[c]=df[c].astype(np.float32)
    return df

print("\nBuilding test features (hidden test wells)..."); t0=time.time()
test_paths=sorted(TEST_DIR.glob('*__horizontal_well.csv'))
print(f"  {len(test_paths)} test wells | {NCPU} threads")
results=Parallel(n_jobs=NCPU,backend='threading',verbose=5)(
    delayed(build_well_test)(
        str(p),str(TEST_DIR/p.name.replace('__horizontal_well.csv','__typewell.csv')))
    for p in test_paths)
ok=[r for r in results if r is not None]
if not ok: raise RuntimeError("All test wells failed feature building.")
test_df=pd.concat(ok,ignore_index=True)
print(f"  test: {test_df.shape}  ({time.time()-t0:.0f}s)")
gc.collect()

# Align to exact feature columns from training
missing=[c for c in feature_cols if c not in test_df.columns]
if missing:
    print(f"  WARNING: {len(missing)} features missing in test, filling with 0: {missing[:5]}")
    for c in missing: test_df[c]=np.float32(0.)
Xt=test_df[feature_cols].astype(np.float32)
Xt_sk=np.nan_to_num(Xt.values,nan=0.)   # HGB needs no NaN
print(f"  Feature matrix: {Xt.shape}")

# ── STEP 3: Load models and predict on hidden test ─────────────────────────────
print("\nLoading models and predicting...")
model_dirs=sorted((ARTS/"models").glob("*/"))
model_preds={}    # name → avg test prediction over folds

for model_dir in model_dirs:
    name=model_dir.name
    fold_files=sorted(model_dir.glob("fold_*.txt")) \
             or sorted(model_dir.glob("fold_*.cbm")) \
             or sorted(model_dir.glob("fold_*.pkl"))

    if not fold_files:
        print(f"  SKIP {name}: no fold files found"); continue

    tp=np.zeros(len(test_df),np.float32)
    n_folds=len(fold_files)
    ext=fold_files[0].suffix

    for fold_file in fold_files:
        fold=int(fold_file.stem.split('_')[1])
        if ext=='.txt':
            m=lgb.Booster(model_file=str(fold_file))
            tp+=m.predict(Xt.values).astype(np.float32)/n_folds
        elif ext=='.cbm':
            m=CatBoostRegressor(); m.load_model(str(fold_file))
            tp+=m.predict(Xt.values).astype(np.float32)/n_folds
        elif ext=='.pkl':
            m=joblib.load(fold_file)
            tp+=m.predict(Xt_sk).astype(np.float32)/n_folds
        else:
            print(f"  Unknown format: {fold_file}"); continue

    model_preds[name]=tp
    print(f"  {name}: {n_folds} folds loaded")

if not model_preds:
    raise RuntimeError("No model predictions could be loaded.")
print(f"  Loaded {len(model_preds)} models: {list(model_preds.keys())}")

# ── STEP 4: Apply ensemble weights ─────────────────────────────────────────────
print(f"\nApplying ensemble ({hc_meta['method']})...")
# Align model predictions to the same order as during training
model_names=hc_meta['model_names']

# Handle case where inference has same or subset of training models
available=[m for m in model_names if m in model_preds]
missing_m=[m for m in model_names if m not in model_preds]
if missing_m:
    print(f"  WARNING: models missing from artifacts: {missing_m}")

if hc_meta['method']=='hillclimb':
    # Use saved weights for available models only
    name_to_idx={n:i for i,n in enumerate(model_names)}
    w_full=hc_meta['weights']
    St_avail=np.column_stack([model_preds[m] for m in available])
    w_avail =np.array([w_full[name_to_idx[m]] for m in available],dtype=np.float64)
    final_test=(St_avail.astype(np.float64)@w_avail).astype(np.float32)
elif hc_meta['method']=='ridge':
    coef=hc_meta['coef']
    St_avail=np.column_stack([model_preds[m] for m in available])
    w_avail =np.array([coef[name_to_idx[m]] for m in available],dtype=np.float64)
    final_test=(St_avail.astype(np.float64)@w_avail).astype(np.float32)
else:
    # Simple average
    final_test=np.stack(list(model_preds.values()),0).mean(0).astype(np.float32)

print(f"  Ensemble prediction shape: {final_test.shape}")

# ── STEP 5: Post-processing ────────────────────────────────────────────────────
def apply_pp(df,model_d,pf_d,alpha,tau,w_pf):
    d=model_d*(1-w_pf)+pf_d*w_pf
    if tau and float(tau)>0:
        d=d*(1.-np.exp(-np.maximum(df['md_since'].values,0.)/float(tau)))
    return d*alpha

def sg_smooth(df,col,sg_w=17,sg_p=3):
    df=df.copy()
    for _,grp in df.groupby('well',sort=False):
        v=grp[col].values; n=len(v); wl=min(sg_w,n)
        if wl%2==0: wl-=1
        if wl>=sg_p+2: v=savgol_filter(v,wl,sg_p)
        df.loc[grp.index,col]=v
    return df

print(f"\nPost-processing: alpha={ALPHA} tau={TAU} w_pf={W_PF}")
pf_test=test_df['pf_ancc'].values - test_df['last_known_tvt'].values
test_df2=test_df.copy()
test_df2['pred']=test_df2['last_known_tvt'].values + \
    apply_pp(test_df2, final_test, pf_test, ALPHA, TAU, W_PF)
test_df2=sg_smooth(test_df2,'pred')

# ── STEP 6: Generate submission ────────────────────────────────────────────────
sample=pd.read_csv(SAMPLE)
sub=sample[['id']].merge(test_df2[['id','pred']].rename(columns={'pred':'tvt'}),on='id',how='left')

# Fallback for any unmatched IDs — use mean delta from known anchor
fallback=float(test_df['last_known_tvt'].mean()) + float(test_df['pf_ancc_delta'].mean())
n_missing=sub['tvt'].isna().sum()
if n_missing>0:
    print(f"  WARNING: {n_missing} IDs not matched — filling with fallback {fallback:.2f}")
sub['tvt']=sub['tvt'].fillna(fallback)

sub[['id','tvt']].to_csv(OUT,index=False)
print(f"\n✅  submission.csv saved → {OUT}  ({len(sub)} rows)")
print(f"    TVT range: [{sub['tvt'].min():.1f}, {sub['tvt'].max():.1f}]")
print(f"    TVT mean:  {sub['tvt'].mean():.2f}")
print(sub.head(8).to_string(index=False))